In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS olist_")

In [ ]:
from pyspark.sql.types import *

orders = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True),
])
order_items = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), False),
    StructField("seller_id", StringType(), False),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])
customers = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), False),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])
products = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True),
])
sellers = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
])
payments = StructType([
    StructField("order_id", StringType(), False),
    StructField("payment_sequential", IntegerType(), False),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True),
])
reviews = StructType([
    StructField("review_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", StringType(), True),
    StructField("review_answer_timestamp", StringType(), True),
])
geolocation = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
])
category_translation = StructType([
    StructField("product_category_name", StringType(), False),
    StructField("product_category_name_english", StringType(), True),
])

In [ ]:
from pyspark.sql.functions import *

base_path = "abfss://data@onelake.dfs.fabric.microsoft.com/olist.Lakehouse/Files/bronze/"

datasets = [
    ("olist_orders_dataset.csv", "bronze_orders", orders),
    ("olist_order_items_dataset.csv", "bronze_order_items", order_items),
    ("olist_customers_dataset.csv", "bronze_customers", customers),
    ("olist_products_dataset.csv", "bronze_products", products),
    ("olist_sellers_dataset.csv", "bronze_sellers", sellers),
    ("olist_order_payments_dataset.csv", "bronze_payments", payments),
    ("olist_order_reviews_dataset.csv", "bronze_reviews", reviews),
    ("olist_geolocation_dataset.csv", "bronze_geolocation", geolocation),
    ("product_category_name_translation.csv", "bronze_category_translation", category_translation),
]

for file_name, table_name, schema in datasets:
    df = (
        spark.read
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema)
        .csv(base_path + file_name)
        .withColumn("ingestion_ts", current_timestamp())
        .withColumn("source_file", input_file_name())
    )

    df.write.format("delta").mode("overwrite").saveAsTable(f"olist_.{table_name}")
    print(f"Created {table_name}")